## Setup

In [ ]:
import sys                                                                                                              
import pandas as pd                                                                                                     

sys.path.append("/home/akubaney/projects/na_mpnn/evaluation")                                                           
from na_eval_utils import calculate_secondary_structure_stats

## Per-Puzzle References

In [ ]:
# Important notes about the reference data.
# - 'struct_dbn' is the DSSR-extracted secondary structure from the competition-
#   provided structure.
# - 'target_dbn' is the competition-provided target secondary structure.
# - 'source' is the source of the structure
# - 'other' is the name of the structure.
openknot_struct_compare_dict = {
	'W01': {
		'struct_dbn': '..((......)).....(((((((((.(([[[[[))))))))))).(((((((((..(....).))).)))))).....]]]]]....',
		'target_dbn': '(.((......))..).((((((((((.(([[[[[))))))))))))(((((((((..(....).))).)))))).....]]]]]....',
		'source': ['RCSB'] ,
		'other':  ['6XRZ SARS-CoV-2 FSE'],
	},
	'W02': {
		'struct_dbn': '((((....((((((((((((......(((((((...[[[[[...))))))).....((....))))).))))))))).]]]]].)))).',
		'target_dbn': '((((....((((((((((((......(((((((...[[[[[...))))))).....((....))))).))))))))).]]]]].)))).',
		'source': ['RCSB'] ,
		'other':  ['7KD1 THF riboswitch'],
	},
	'W03': {
		'struct_dbn': '(((((..((......(((((((((.....{[[[[[[[.)))))))))...))....]]]]]}.]]..)))))..',
		'target_dbn': '(((((.(((......(((((((((..(..{[[[[[[[))))))))))...)))...]]]]]}.]]..)))))..',
		'source': ['RCSB'] ,
		'other':  ['3Q3Z c-di-GMP-II switch'],
	},
	'W04': {
		'struct_dbn': '(((((((((((.(((((..(((((....)))).))))))[))((((..].))))((..[[[[[.))))))))))).]]]]]...',
		'target_dbn': '(((((((((((.((((((.[((((....)))))])))))[))((((..].))))((..[[[[[.))))))))))).]]]]]...',
		'source': ['RCSB'] ,
		'other':  ['3T4B HCV IRES'],
	},
	'W05': {
		'struct_dbn': '.(((((((((((((((((((..[[[[[[.)))))(((....)))(((....)))))))))))))))))((((((..]]]]]].)))))).',	
		'target_dbn': '.(((((((((((((((((((..[[[[[[.)))))(((....)))(((....)))))))))))))))))((((((..]]]]]].)))))).',
		'source': ['RCSB'] ,
		'other':  ['2AE KL'],
	},
	###
    # W06 and W07 had 3 and 2 missing residues in the corresponding PDB
    # structures; with the rest of the structure held constant, RFDpoly was
    # used to generate the structure of the 2 missing residues.
	'W06': {
		'struct_dbn': '[[[[......((((((...]]]]((((((((..........))))))))....((((((((((........))))))).)))..))))))..........',	
		'target_dbn': '((((......[[[[[[...))))(((((((((........)))))))))....((((((((((........))))))).)))..]]]]]]..........',
		'source': ['RCSB'] ,
		'other':  ['W06__1720455712_BFF_3.00__SSG_cleaned_01_4__pSSA_0.870'],
	},
	'W07': {
		'struct_dbn': '.[[[[[..........((((((((....))))))))(((((((((((..]]]]]...........)))))))))))',	
		'target_dbn': '.[[[[[..........((((((((....))))))))(((((((((((..]]]]]...........)))))))))))',
		'source': ['RCSB'] ,
		'other':  ['W07__1720455712_BFF_3.00__SSG_cleaned_01_4__pSSA_0.921'],
	},
    ###
	'W08': {
		'struct_dbn': '[[[[[......(.((((((((((]]]]].....).).)))))))).).',	
		'target_dbn': '[[[[[......((((((((((((]]]]].....).).)))))))))).',
		'source': ['RCSB'] ,
		'other':  ['2M8K telomerase'],
	},
	'W14': {
		'struct_dbn': '(((((....((((....))))(((..((((((.[[[[[.))).))).)))..(((((....)))))))))).((((....)))).......]]]]].',	
		'target_dbn': '(((((....((((....))))(((..((((((.[[[[[.))).))).)))..(((((....)))))))))).((((....)))).......]]]]].',
		'source': ['RCSB'] ,
		'other':  ['4OQU SAM I/IV switch'],
	},
	'W15': {
		'struct_dbn': '((((((((((((((((((((((...[[[[.))))))).).)))).))))).((..........)).....)))))..........]]]]',	
		'target_dbn': '((((((((((((((((((((((...[[[[.))))))).).)))).))))).((..........)).....)))))..........]]]]',
		'source': ['RCSB'] ,
		'other':  ['7KGA donggang dumbbell'],
	},
	'W16': {
		'struct_dbn': '(((((((...[[[[[[(((.[[.....))))))))))]]....((((..........)))).....]]]]]]',	
		'target_dbn': '(((((((...[[[[[[(((.[[.....))))))))))]]....((((..........)))).....]]]]]]',
		'source': ['RCSB'] ,
		'other':  ['1DRZ HDV ribozyme'],
	},
	'W17': {
		'struct_dbn': '[[[[[[...((((((((((.......))).]]]]]]..(((((..........)))))....)))))))',	
		'target_dbn': '[[[[[[...((((((((((.......))).]]]]]]..(((((..........)))))....)))))))',
		'source': ['RCSB'] ,
		'other':  ['7QR4 CPEB3'],
	},
	'P01': {
		'struct_dbn': '.[[[[..((((((]]]]........................)))))).....',	
		'target_dbn': '.[[[[{.((((((]]]]......(((((....)))))}.[.))))))]....',
		'source': ['trRosetta'] ,
		'other':  ['Thermotoga_petrophila_fluoride_riboswitch'],
	},
	'P02': {
		'struct_dbn': '......((.(.(..(...[[[.....((......))....)..).).))..........(((..]]]....))).',	
		'target_dbn': '.....(((((((..(.[[[[[...((((......))))..)..))))))).........(((..]]]]]..))).',
		'source': ['trRosetta'] ,
		'other':  ['ZTP_switch'],
	},
	'P03': {
		'struct_dbn': '.((((((((..(.[[[[[....((((....))))..)..))))))))........(((..]]]]]..)))...',	
		'target_dbn': '.((((((((..(.[[[[[....((((....))))..)..))))))))........(((..]]]]]..)))...',
		'source': ['RNASolo'] ,
		'other':  ['pfl_switch'],
	},
	'P04': {
		'struct_dbn': '....................((((((((((((....))))))))))))..[[[[...((((..]]]]..((((.......)))).....))))',	
		'target_dbn': '....................((((((((((((....))))))))))))..((((...[[[[..))))..((((.......)))).....]]]]',
		'source': ['RCSB'] ,
		'other':  ['sgRNA'],
	},
	'P05': {
		'struct_dbn': '((((((((.....[[[[[[.)))))))).........((((....))))..]]]]]]..',	
		'target_dbn': '((((((((.....[[[[[[.)))))))).........((((....))))..]]]]]]..',
		'source': ['RCSB'] ,
		'other':  ['PreQ1-II_switch'],
	},
	'P06': {
		'struct_dbn': '.[[[[.....((((.......))))(((((.]]]]((((((....))))))(((((((.....))))))))))))',	
		'target_dbn': '.((((.....((((.......))))[[[[[[))))((((((....))))))..................]]]]]]',
		'source': ['trRosetta'] ,
		'other':  ['Grapevine_leafroll-associated_virus_-_2'],
	},
	'P07': {
		'struct_dbn': '...((((......[))))........(((((..((.((.(......).)).))..))))).....]..............',	
		'target_dbn': '..(((((..[[[[[))))).......((((((.((((((((....)))))))).)))))).....]]]]]..........',
		'source': ['trRosetta'] ,
		'other':  ['Turdivirus_3'],
	},
	'P08': {
		'struct_dbn': '..((((.((..(((((...[[[[[..)))))....((((((......)))))))).))))..(((.((.....)).)))........]]]]]....',	
		'target_dbn': '.((((((((..((((((..[[[[[.))))))....((((((......)))))))))))))).((((((.....))))))........]]]]]....',
		'source': ['trRosetta'] ,
		'other':  ['Poa_semilatent_virus'],
	},
	'P09': {
		'struct_dbn': '..[[....(((((((.((((((((((((.....))))))))))))...((..]]))....))))))).',	
		'target_dbn': '..((....[[[[[[..((((((((((((.....))))))))))))...[[..))]].....]]]]]].',
		'source': ['FARFAR2'] ,
		'other':  ['E._coli'],
	},
	'P10': {
		'struct_dbn': '.(((((............[[[[..............)))))......(((((.....)))))......]]]]....',	
		'target_dbn': '(((((((......[[[.[[[[[(((.[[....))))))))))]]...(((((.....)))))......]]]]]]]]',
		'source': ['FARFAR2'] ,
		'other':  ['Diplonema_papillatum'],
	},
	'P11': {
		'struct_dbn': '................................((((((.((((((......(.((......).))........))))))...............))))))',	
		'target_dbn': '(((((((((((.(((......))).)))))))[[[[[[((((((((..(((((((......))))...))).))))))))...)))).......]]]]]]',
		'source': ['trRosetta'] ,
		'other':  ['PN.v282'],
	},
	'P12': {
		'struct_dbn': '......[[[[[..((((((....))))))[[[[[....((((((((((((((((....))))))).....]]]]]...]]]]]..)))))))))......',	
		'target_dbn': '......((((((.((((((....))))))(((((....[[[[[[[[[(((((((....))))))).....)))))..))))))..]]]]]]]]]......',
		'source': ['trRosetta'] ,
		'other':  ['mod_of_f67'],
	},
	'P13': {
		'struct_dbn': '(((((((..[[[[[[)))))))..((..((....))..))..(((((.........)))))...(((((((...........]]]]]]..)))))))...',	
		'target_dbn': '(((((((..[[[[[[)))))))..((..((....))..))..(((((.........)))))...(((((((...........]]]]]]..)))))))...',
		'source': ['FARFAR2'] ,
		'other':  ['UC2414'],
	},
	'P14': {
		'struct_dbn': '..(((((((((.[[[..[[....(....)............))))))))).......(((((((.(.....]]..]]].......)))))))).......',	
		'target_dbn': '.(((((((((([[[[[[[[(((((....)))))........))))))))))......((((((((((((..]]]]]]]]..)))))))))))).......',
		'source': ['trRosetta'] ,
		'other':  ['SV_j116'],
	},
	'P15': {
		'struct_dbn': '(((((..[[[[.[[)))))...[.[[[[[..((((((]]]]].]..]].]]]]............(((((.((......)).)))))......)))))).',	
		'target_dbn': '(((((.[[[[[[[[)))))...(((((((..{{{{{{)))))))..]]]]]]]].......(((((((((((........)))))))))))..}}}}}}.',
		'source': ['trRosetta'] ,
		'other':  ['UC2458'],
	},
	'P16': {
		'struct_dbn': '.(.((((..........)))).)..............(..(............[..[[.)..)..((.]]..].........[.))........].....',	
		'target_dbn': '.(((((([[[[[[[[[[))))))...((((((]]]]][[[[[))))))....((((((]]]]][[[[[))))))...((((((]]]]]]]]]])))))).',
		'source': ['trRosetta'] ,
		'other':  ['AK_PK100-3'],
	},
	'P17': {
		'struct_dbn': '(((.(....((((..(((((((((.....)))((((((...[[[[.))))))(((......]]]][[[[[[[..)))))))))))))..]]]]]]]))))',	
		'target_dbn': '(((......((((..(((((((((.....)))((((((...[[[[.))))))(((......]]]][[[[[[[..)))))))))))))..]]]]]]].)))',
		'source': ['FARFAR2'] ,
		'other':  ['ZG21'],
	},
	'P18': {
		'struct_dbn': '.((.((((.....((..(((..((.....{.{{))..)))..))........(.[[.[...)..........)))).))....}}.}......].]]...',	
		'target_dbn': '.(((((((((...(((((((((((..[[[[[[.)))))))))))..(((((((.{{{{{{.)))))))..))))))))).....]]]]]].}}}}}}...',
		'source': ['trRosetta'] ,
		'other':  ['Terminal_3'],
	},
	'P19': {
		'struct_dbn': '[[[[[[[[.[[[[[[[[[.....((((((((((.(((((((((]]]]]]]]].))))))))).((((((((.]]]]]]]])))))))).)))))))))).',	
		'target_dbn': '((((((((((((((((((.....[[[[[[[[[[.[[[[[[[[[))))))))))]]]]]]]]].[[[[[[[[.))))))))]]]]]]]].]]]]]]]]]].',
		'source': ['FARFAR2'] ,
		'other':  ['SV_r7_100_a7'],
	},
	'P20': {
		'struct_dbn': '(((((((.((((((..))))))((((....)))).[[[[[.))))))).((((((((.]]]]](((((..)))))((((((....)))))).))))))))',	
		'target_dbn': '(((((((.(((((....)))))((((....)))).[[[[[.))))))).((((((((.]]]]]((((....))))((((((....)))))).))))))))',
		'source': ['FARFAR2'] ,
		'other':  ['Kissing_multiloops'],
	},
	'Q01': {
		'struct_dbn': '.((((.(...(.(((((..(.(.((....((((...(((.....)))...))))......)).).)..((((....)))).....)))))...)..(((.[[[[..)))).)))).(((((((((.(((.....))).)))))))))...]]]]....................',	
		'target_dbn': '.((((.(...(.(((((..(.(.((....((((...(((.....)))...))))......)).).)..((((....)))).....)))))...)..(((.[[[[..)))).)))).(((((((((..((.....))..)))))))))...]]]]....................',
		'source': ['RNSolo'] ,
		'other':  ['CRISPR_Guide_7YOJ'],
	},
	'Q02': {
		'struct_dbn': '(((((((((((.......((((((.....((((((..[[[[[[.)))))))))))))))))))))))((((((((((.]]]]]].((((((.....((((((.........)))))))))))).))))))))))',	
		'target_dbn': '(((((((((((.......((((((.....((((((..[[[[[[.)))))))))))))))))))))))((((((((((.]]]]]].((((((.....((((((.........)))))))))))).))))))))))',
		'source': ['RNSolo'] ,
		'other':  ['Nanobracelet_7JRT'],
	},
	'Q03': {
		'struct_dbn': '(((((..........))))).......[[[...((((((..]]]..............))))))((((((.[[[[[[))))))...(((((((((.....(((((((....)))))))......))))))))).]]]]]]',	
		'target_dbn': '(((((..........))))).......[[[...((((((..]]]..............))))))((((((.[[[[[[))))))...(((((((((.....(((((((....)))))))......))))))))).]]]]]]',
		'source': ['RNSolo'] ,
		'other':  ['GLMS_ribozyme_3G9C'],
	},
	'Q04': {
		'struct_dbn': '(((.....(((((.(..).)))))..)))(((((....(.((((.....))))...)....)))))(((...[[[))).....]]].........[..(((((((....)))))))..((((((......](((((((.(((((....))))))))))))))))))',	
		'target_dbn': '(((......((((......))))...)))(((((......((((.....))))........)))))(((...[[[))).....]]].........{..(((((((....)))))))..((((((......}(((((((.(((((....))))))))))))))))))',
		'source': ['FARFAR2'] ,
		'other':  ['Tuberculosis_T-box_6UFG'],
	},
	'Q05': {
		'struct_dbn': '((((((((...[[[[[[.))))))))...............[[[[[(...).(.((((((((((((((..........)))))))..((((.]]]]]))))((.(((((....))))).)))))))))).]]]]]].',	
		'target_dbn': '((((((((...[[[[[[.))))))))...............[[[[[(...).(.((((((((((((((..........)))))))..((((.]]]]]))))((.((((......)))).)))))))))).]]]]]].',
		'source': ['FARFAR2'] ,
		'other':  ['Ligase_ribozyme_3HHN'],
	},
	'Q06': {
		'struct_dbn': '...(((((((..............(((.(((......[[..[..[.....))).)))...[.....)))))))..[...................(..]..........].......).....]..]..]]........(((.((..[.[.[.))..)))..............................].].]......',	
		'target_dbn': '...(((((((..............(((((........[[[[[..........))))).........)))))))[[[[.(..........)..(((((]]]](.((.....)).)..))))).....]]]]]..(((((.(((((((.[[[[[)))).))).((((((....)))))))))))........]]]]]......',
		'source': ['trRosetta'] ,
		'other':  ['Taura_IRES_5JUP'],
	},
	'Q07': {
		'struct_dbn': '..[[[..[[..[[.....(((((((....]]......(((((..(.((.....)).)...)..)))).]]..]]](((((((((....))))))))).........)))))))...',	
		'target_dbn': '..(((((((..((((((..[[[[[[)))))).....(((((.((((((.....))))))....))))))))))))(((((((((....))))))))).........]]]]]]....',
		'source': ['trRosetta'] ,
		'other':  ['Broad_bean_mottle_virus_PKB135'],
	},
	'Q08': {
		'struct_dbn': '........((.(((((((((((.............[[[[[[..((.(..(((((......)))))).))..((((((....))))))......))))))))).)).))...........]]]]]]...',	
		'target_dbn': '........((((((((((((((...........[[[[[[[[((((((..(((((......)))))))))))((((((....))))))......))))))))).)))))...........]]]]]]]].',
		'source': ['trRosetta'] ,
		'other':  ['Rous_sarcoma_virus_PKB174'],
	},
	'Q09': {
		'struct_dbn': '.(((.(((...((((((.....((((((....)))).))[[[[[[[[))))))))).)))....(((.(((((((((((.......))))(((((((....)))))))..)))))))))).....((.(((((]]]]]]]].......(.(((....))).).....))))))).',	
		'target_dbn': '.(((.(((...((((((.....((((((....)))).))[[[[[[[[))))))))).)))....(((.(((((((((((.......))))(((((((....)))))))..))))))))))........(((((]]]]]]]].........(((....))).......)))))...',
		'source': ['FARFAR2'] ,
		'other':  ['Methylosinus-1_RF03108'],
	},
	'Q10': {
		'struct_dbn': '.(((....(((((((((((.(((((((((((((((((.(((((((([[[[[[.............))))((((((((....))))))))....(((((..............)))))(((((.(((..........)))..)))))..))))....))))))))......)))))))))..)))]]]]]].))))))))...))).',	
		'target_dbn': '.(((....(((((((((((.((((((((((((((((..(((((((([[[[[[.............))))((((((((....)))))))).....((((..............)))).(((((.(((..........)))..)))))..)))).....)))))))......)))))))))..)))]]]]]].))))))))...))).',
		'source': ['FARFAR2'] ,
		'other':  ['SCARNA2_URS00008E39F0_9606'],
	},
	'Q11': {
		'struct_dbn': '..(((((((((((.[[.[[[[[..((((((.((((........(((((((((((((...........)))))))))))))........)))).))))).)........))))))))))).((((((((((]]]]].]].(((((((((.((((....................................................)))).)))))))))...........))))))))))',
		'target_dbn': '..((((((((((([[[[[[[[[.((((((((((((........(((((((((((((...........)))))))))))))........))))))))))))........))))))))))).((((((((((]]]]]]]]](((((((((.((............((((((((((............))))))))))............)).)))))))))...........))))))))))'	,
		'source': ['trRosetta'] ,
		'other':  ['SV_g'],
	},
	'Q12': {
		'struct_dbn': '...((((((((.(............[.[[[[..).)))))))).......((((.(...]]]].]..............).)))).......(((((((...........................))))))).........................((((((((...................[[[[[..)))))))).............]]]]]......................',
		'target_dbn': '...((((((((((............[[[[[[[[))))))))))....((((((((((]]]]]]]].[[[[[[[[[[[[))))))))))....((((((((((]]]]]]]]]]]][[[[[[[[[))))))))))....((((((((((]]]]]]]]]..[[[[[[[[[))))))))))....((((((((((]]]]]]]]]............))))))))))....((((....))))..'	,
		'source': ['trRosetta'] ,
		'other':  ['SV_i'],
	},
	'Q13': {
		'struct_dbn': '.((((((.....(((......))).......[[[[[[[[[[[[[.((((((((((((((((.............((((((............(((((((((((((((((...(((((..(.....)..)))))......)))))))))))))))))]]]]]]]]]]]]]..))).)))..)))))))))))))))).)))))).',	
		'target_dbn': '.((((((.......(......).........(((((((((((((.[[[[[[[[[[[[[[[[.............[[[[[[............(((((((((((((((((...(((((...........)))))......))))))))))))))))))))))))))))))..]]].]]]..]]]]]]]]]]]]]]]].)))))).',
		'source': ['FARFAR2'] ,
		'other':  ['SV_h3'],
	},
	'Q14': {
		'struct_dbn': '.((((((((((((((((.......))))))))((((((((((((((.......))))))))((((((((.(..(((((((((((....((......((((((([[[[[[[[[)))))))..))....))))))))))).)..))......(((((((]]]]]]]]])))))))))))))((((((((.......))))))))))))))((((((((.......)))))))))))))))).',
		'target_dbn': '.((((((((((((((((.......))))))))((((((((((((((.......))))))))((((((((....(((((((((((....((......((((((([[[[[[[[[)))))))..))....)))))))))))....))......(((((((]]]]]]]]])))))))))))))((((((((.......))))))))))))))((((((((.......)))))))))))))))).'	,
		'source': ['FARFAR2'] ,
		'other':  ['SV_c'],
	},
	'Q15': {
		'struct_dbn': '...........((((((((((.........))))))))))....(((((..(((((((((.........)))))))))....(((((((((((((((((.........))))))))))).......(.[[[[[...........[..)....)))))).....(.((.(((.........))).)).)...)))))....((((..]...........]]]]]..))))...........',
		'target_dbn': '.((((((((((((((((((((.........))))))))))).(((((((..(((((((((.........)))))))))(((((((((((((((((((((.........))))))))))).....((([[[[[[[[[[[[[[[[...)))...))))))))))(((((((((.........)))))))))..)))))))..(((((...]]]]]]]]]]]]]]]]))))).))))))))).'	,
		'source': ['trRosetta'] ,
		'other':  ['SV_f'],
	},
	'Q16': {
		'struct_dbn': '.((((..((((.(((((.....))))))))).........(((((...........).))))...((((((............))))))....)))).....((((((((((((((.....)))))..((((((.....))))))(((.((((((.[[[[[[.[.)))))))))..)))))))))...((.(((.((((....(...].]]]]]]...).........)))).))).)).',
		'target_dbn': '.((((.(((((.(((((.....))))))))))......((((((((((.....)))).))))))(((((((.(((.....))))))))))...)))).....((((((((((((((.....)))))..((((((.....))))))(((.((((((.[[[[[[[[.)))))))))..)))))))))...(((((((((((..(((((.]]]]]]]].))))).......))))))))))).'	,
		'source': ['trRosetta'] ,
		'other':  ['AK_PK240-3'],
	},
	'Q17': {
		'struct_dbn': '....(((((((((....[[[[...))))))))).(((((((((.......)))))))))..((((((.....))))))...((((((.....[[[[..))))))..((((((........))))))...(((((........))))).(....)...((((......))))...((((((......))))))..(((((.]]]]...))))).(((((((....]]]].)))))))....',
		'target_dbn': '....(((((((((....[[[[...))))))))).(((((((((.......)))))))))..((((((.....))))))...((((((.....[[[[..))))))..((((((........))))))...(((((........)))))..........((((......))))...(((((........)))))..(((((.]]]]...))))).(((((((....]]]].)))))))....'	,
		'source': ['FARFAR2'] ,
		'other':  ['pknot240_17'],
	},
	'Q18': {
		'struct_dbn': '((((((((...(((((((((((.[[[[[[[..)).)..)).)))))).......(.(((((.((((((((((.[[[[.))))))))))...))))).)....)))))))).((((((.((((((((((..(((..((]]]]...)).)))((((.(((((.((((.((((((((..)))))))).((((...(...]]]]]]])))))..))))))))))))))))))))))))))))).',
		'target_dbn': '((((((((...(((((((((((.[[[[[[[..)).)..)).)))))).......(.(((((.((((((((((.[[[[.))))))))))...))))).)....)))))))).((((((.((((((((((..(((..((]]]]...)).)))((((.(((((.((((.(((((((....))))))).((((...(...]]]]]]])))))..))))))))))))))))))))))))))))).'	,
		'source': ['FARFAR2'] ,
		'other':  ['Astros-Eli-mod'],
	},
	'Q19': {
		'struct_dbn': '...............((((((...[......))))))...(((((((((.((((((.((.(((.((.................................(.((.(((((.....]..))))).)).)..)).))).)).((((((((((....[[.......))))))))))....)))))).(((((((((((...]].......))))))))))).))))))))).............',

		'target_dbn': '((((((((((((.(((((((((.[[[[[..))))))))).(((((((((.((((((((((((((((.((((((((((((......)))))))))))).(((((((((((..]]]]].))))))))))).))))))))).((((((((((..[[[[[[.....))))))))))...))))))).(((((((((((.]]]]]].....))))))))))).))))))))).))))))))))))'	,
		'source': ['trRosetta'] ,
		'other':  ['SV_r7_240_4'],
	},
	'Q20': {
		'struct_dbn': '....[[[[[[[....(((((..[[..(((((((........]]..((..((((((...(((((((((.((......))))....))).))))....))))))..)).....)))))))))))).....(.(((((((.(((((((((..((((..[[[[[[.))).)...[[)))))))))...((((((((....]].......]]]]]]]]]]]]]..))))))))..))))))).).',
		'target_dbn': '.......((......(((((..((..[[[[[[[........))..(...((((((...(((((((.(.((......))).....))).))))....))))))...).....]]]]]]]))))).......[[[[[[...((((((.(.....(..[[[[[[.)........[).))))))....{{{{{{.{....]........]]]]]]..)).....}.}}}}}}...]]]]]]...'	,
		'source': ['FARFAR2'] ,
		'other':  ['Crazy_Legs_study_7'],
	},
}

## F1 Computation

In [ ]:
rows = []       
for puzzle_id, entry in openknot_struct_compare_dict.items():
    stats = calculate_secondary_structure_stats(                                                                        
        reference_secondary_structure=entry["target_dbn"],
        subject_secondary_structure=entry["struct_dbn"],                                                                
    )                                                                                                                   
    rows.append({
        "puzzle_id": puzzle_id,                                                                                         
        "f1_score_pairs": stats["f1_score_pairs"],
        "f1_score_loops": stats["f1_score_loops"],
        "source": entry["source"][0],                                                                                   
    })                                                                                  

In [ ]:
df = pd.DataFrame(rows)
df.to_csv("./per_puzzle_consistency.csv", index=False)
df           